## Python Code for Data Preprocessing of Nanomedicine Research Papers

Below is the Python code that performs the specified data preprocessing steps on nanomedicine research papers. The code includes:

1. Text Cleaning:
* Lowercasing
* Removing punctuation and numbers
* Stop words removal
* Lemmatization
2. Handling Special Terms:
* Creating a custom dictionary to preserve nanomedicine-specific terminology
3. De-duplication:
* Removing duplicate entries

In [1]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import json

# Download NLTK data files (only need to run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /home/chris/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/chris/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/chris/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Function to remove duplicates
def remove_duplicates(df):
    # Remove duplicate rows based on the 'title' and 'abstract' columns
    df_dedup = df.drop_duplicates(subset=['title', 'abstract'])
    return df_dedup

In [ ]:
# Function to preprocess text
def preprocess_text(text, custom_terms=None):
    # Convert to lowercase
    text = text.lower()
    
    if custom_terms is not None:
        # Replace special nanomedicine terms with placeholders to preserve them
        for term in custom_terms:
            # Replace spaces in term with underscores
            term_underscore = term.replace(' ', '_')
            # Use regex to replace the term in text
            text = re.sub(r'\b' + re.escape(term.lower()) + r'\b', term_underscore, text)
    
    # Remove punctuation and numbers
    text = re.sub(r'[{}]'.format(string.punctuation), ' ', text)
    text = re.sub(r'\d+', '', text)

    # Tokenize the text
    tokens = text.split()
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    if custom_terms is not None:
        # Replace placeholders back to original terms with spaces
        tokens = [word.replace('_', ' ') if word in [t.replace(' ', '_') for t in custom_terms] else word for word in tokens]

    # Join tokens back into a string
    processed_text = ' '.join(tokens)
    
    return processed_text

In [12]:
data_dict = json.load(open('raw_dataset.json', 'r'))

In [18]:
for key, value in data_dict.items():
    print(key, value)
    print(value['Abstract'])
    break



WOS:001291133100001 {'Publication Type': 'J', 'Authors': 'Gharibkandi, NA; Wawrowicz, K; Walczak, R; Majkowska-Pilip, A; Wierzbicki, M; Bilewicz, A', 'Book Authors': nan, 'Book Editors': nan, 'Book Group Authors': nan, 'Author Full Names': 'Gharibkandi, Nasrin Abbasi; Wawrowicz, Kamil; Walczak, Rafal; Majkowska-Pilip, Agnieszka; Wierzbicki, Mateusz; Bilewicz, Aleksander', 'Book Author Full Names': nan, 'Group Authors': nan, 'Article Title': '109Pd/109mAg in-vivo generator in the form of nanoparticles for combined β- - Auger electron therapy of hepatocellular carcinoma', 'Source Title': 'EJNMMI RADIOPHARMACY AND CHEMISTRY', 'Book Series Title': nan, 'Book Series Subtitle': nan, 'Language': 'English', 'Document Type': 'Article', 'Conference Title': nan, 'Conference Date': nan, 'Conference Location': nan, 'Conference Sponsor': nan, 'Conference Host': nan, 'Author Keywords': 'Pd-109/Ag-109m in-vivo generator; Hepatocellular carcinoma; Auger electron therapy; Nanotechnology', 'Keywords Plus

In [19]:
json_files = json.load(open('raw_dataset.json', 'r'))

In [24]:
new_json_files = {}
for key, data  in json_files.items():
    # convert list of abstracts into a single string

    cleaned_text = str(data['Article Title']) + ' ' + str(data['Abstract'])
    cleaned_abstract = preprocess_text(cleaned_text)

    # append title to abstract
    data['cleaned_text'] = cleaned_abstract


In [27]:
for key, value in json_files.items():
    print(key, value)
    print(value['Abstract'])
    print(value['Article Title'])
    print(value['cleaned_text'])
    break

WOS:001291133100001 {'Publication Type': 'J', 'Authors': 'Gharibkandi, NA; Wawrowicz, K; Walczak, R; Majkowska-Pilip, A; Wierzbicki, M; Bilewicz, A', 'Book Authors': nan, 'Book Editors': nan, 'Book Group Authors': nan, 'Author Full Names': 'Gharibkandi, Nasrin Abbasi; Wawrowicz, Kamil; Walczak, Rafal; Majkowska-Pilip, Agnieszka; Wierzbicki, Mateusz; Bilewicz, Aleksander', 'Book Author Full Names': nan, 'Group Authors': nan, 'Article Title': '109Pd/109mAg in-vivo generator in the form of nanoparticles for combined β- - Auger electron therapy of hepatocellular carcinoma', 'Source Title': 'EJNMMI RADIOPHARMACY AND CHEMISTRY', 'Book Series Title': nan, 'Book Series Subtitle': nan, 'Language': 'English', 'Document Type': 'Article', 'Conference Title': nan, 'Conference Date': nan, 'Conference Location': nan, 'Conference Sponsor': nan, 'Conference Host': nan, 'Author Keywords': 'Pd-109/Ag-109m in-vivo generator; Hepatocellular carcinoma; Auger electron therapy; Nanotechnology', 'Keywords Plus

In [29]:
# delete files for which the abstract is empty or the title is empty
for key, value in json_files.items():
    if value['Abstract'] == '' or value['Article Title'] == '':
        del json_files[key]

In [30]:
# check length of json_files
print(len(json_files))

67160


In [31]:
# write the cleaned data to a new json file
with open('cleaned_dataset.json', 'w') as f:
    json.dump(json_files, f, indent=4)

In [4]:
# delete files for which the abstract is "Not Available"
count = 0
json_files = [f for f in os.listdir('embeddings_subset') if f.endswith('.json')]
for file in json_files:
    with open(os.path.join('embeddings_subset', file), 'r') as f:
        data = json.load(f)
        if data['abstract'] == 'Not Available':
            count += 1
            os.remove(os.path.join('embeddings_subset', file))

print(count)

30
